# Interagir avec les LLMs et Prompt Engineering avec LangChain
Objectif: Analyser les sentiments d'avis clients en français en utilisant différentes 
techniques de prompting (Zero-shot, Few-shot, Chain-of-Thought) avec LangChain.

In [ ]:
import os
import os.path as osp

from pathlib import Path
from pprint import pprint

import sys

root = Path.cwd().parent 
if str(root) not in sys.path:    
    sys.path.append(str(root))

In [ ]:
from src.config import CFG

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [ ]:
model = ChatOpenAI(
    base_url=CFG.BASE_URL,
    api_key=CFG.API_KEY,
    model=CFG.BASE_MODEL,  # Exemple de modèle ouvert
    temperature=CFG.GENERATION_TEMPERATURE
)

In [ ]:
reviews = [
    "Service client catastrophique, personne n'a su résoudre mon problème de facturation !",
    "Installation rapide de la fibre, technicien très professionnel et courtois. Top !",
    "La connexion mobile coupe sans cesse dans mon quartier, c'est insupportable."
]

## TECHNIQUE 1 : Zero-Shot Prompting

In [ ]:

zero_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un expert en analyse de la relation client pour les télécoms. \
     Classe l'avis client suivant en trois catégories : Sentiment (Positif/Négatif/Neutre), Thématique (Facturation/Technique/Réseau), et Urgence (Faible/Moyenne/Haute). \
     Réponds sous forme de bullet points."),
    ("human", "Avis client : {review}")
])

chain_zero = zero_shot_prompt | model | StrOutputParser()

for review in reviews[:1]:
    result = chain_zero.invoke({"review": review})
    print(f"Avis : {review}\nRésultat:\n{result}\n")



## TECHNIQUE 2 : Few-Shot Prompting

In [ ]:
# Exemples de référence pour guider le modèle
examples = [
    {"review": "La facture est trop élevée ce mois-ci.", "sentiment": "Négatif", "thematique": "Facturation"},
    {"review": "Merci pour l'aide, tout fonctionne parfaitement.", "sentiment": "Positif", "thematique": "Service"}
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "Avis : {review}"),
    ("ai", "Sentiment: {sentiment}\nThématique: {thematique}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant e-commerce et télécoms. Analyse l'avis en suivant le format des exemples."),
    few_shot_prompt,
    ("human", "Avis : {review}")
])

chain_few = final_few_shot_prompt | model | StrOutputParser()

for review in reviews[1:2]:
    result = chain_few.invoke({"review": review})
    print(f"Avis : {review}\nRésultat:\n{result}\n")



# TECHNIQUE 3 : Chain-of-Thought (Chaîne de pensée)

In [ ]:
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un analyste qualité. Pour analyser l'avis client, procède étape par étape :\n"
               "Étape 1 : Identifie les mots-clés du texte.\n"
               "Étape 2 : Déduis l'impact émotionnel sur le client.\n"
               "Étape 3 : Conclus sur le sentiment final (Positif, Négatif, Neutre)."),
    ("human", "Avis client : {review}")
])

chain_cot = cot_prompt | model | StrOutputParser()

for review in reviews[2:]:
    result = chain_cot.invoke({"review": review})
    print(f"Avis : {review}\nRésultat détaillé:\n{result}\n")